# ShrimpDiseaseImageBD — Pure Object Detection YOLO Benchmark under 12M — RTX4090

**Version:** `v3_FIXED_OD_ONLY`

This notebook fixes the previous `Path(None)` / `HEALTHY_DIR=None` failure.

Current protocol:
- Train **pure object detection only** on annotated diseased folders: `1. BG`, `2. WSSV`, `4. WSSV_BG`.
- Do **not** add Healthy images to the OD dataset.
- Healthy will be handled later by a separate classification gate/head.
- OD classes are canonical symptom-level boxes:
  - `0 = BG`
  - `1 = WSSV`

Critical rule:
```python
USE_HEALTHY_NEGATIVES = False
HEALTHY_DIR = None
```

No later cell may call `list_images(HEALTHY_DIR)` or `Path(HEALTHY_DIR)` unless guarded.

In [1]:
from pathlib import Path
import os, sys, json, time, shutil, random, subprocess, platform, traceback, csv, math
from datetime import datetime

WORKDIR = Path("/home/drnguyenvinh/notebooks/shrimp_od_yolo_ultralytics_under12M_17models_rtx4090_v3_FIXED_OD_ONLY")
WORKDIR.mkdir(parents=True, exist_ok=True)

KAGGLE_DATASET = "nhanayai/shrimpdiseaseimagebd"

LOCAL_RAW = WORKDIR / "kaggle_raw" / "shrimpdiseaseimagebd"
YOLO_DATASET_DIR = WORKDIR / "datasets" / "shrimp_od_yolo_disease_only_2cls"
RUNS_DIR = WORKDIR / "runs_train_30ep"
EVAL_DIR = WORKDIR / "runs_eval_test"
TABLES_DIR = WORKDIR / "tables"
ERROR_DIR = WORKDIR / "errors"

for p in [LOCAL_RAW, YOLO_DATASET_DIR, RUNS_DIR, EVAL_DIR, TABLES_DIR, ERROR_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SEED = 42
TRAIN_RATIO = 0.80
VAL_RATIO = 0.10
TEST_RATIO = 0.10

EPOCHS = 30
IMG_SIZE = 1024
DEVICE = 0
WORKERS = 8

# Pure OD protocol:
USE_HEALTHY_NEGATIVES = False
HEALTHY_DIR = None

print("WORKDIR:", WORKDIR)
print("Python:", sys.version)
print("Platform:", platform.platform())
print("USE_HEALTHY_NEGATIVES:", USE_HEALTHY_NEGATIVES)
print("HEALTHY_DIR:", HEALTHY_DIR)

WORKDIR: /home/drnguyenvinh/notebooks/shrimp_od_yolo_ultralytics_under12M_17models_rtx4090_v3_FIXED_OD_ONLY
Python: 3.13.2 | packaged by Anaconda, Inc. | (main, Feb  6 2025, 18:56:02) [GCC 11.2.0]
Platform: Linux-6.14.0-37-generic-x86_64-with-glibc2.39
USE_HEALTHY_NEGATIVES: False
HEALTHY_DIR: None


## 1. Minimal dependency check

This cell installs only missing **required** packages:
- `ultralytics`
- `kagglehub`

It does not upgrade Torch, NumPy, OpenCV, Pandas, Matplotlib, or Scikit-learn.

In [2]:
import importlib.util
import subprocess
import sys

def is_installed(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is not None

def pip_install_minimal(packages):
    if not packages:
        print("All required packages already installed. No pip install needed.")
        return
    cmd = [sys.executable, "-m", "pip", "install"] + packages
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)

missing_required = []
if not is_installed("ultralytics"):
    missing_required.append("ultralytics")
if not is_installed("kagglehub"):
    missing_required.append("kagglehub")

pip_install_minimal(missing_required)

import torch
import ultralytics
from ultralytics import YOLO

print("Ultralytics:", ultralytics.__version__)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))
    print("CUDA capability:", torch.cuda.get_device_capability(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))

All required packages already installed. No pip install needed.
Ultralytics: 8.4.67
Torch: 2.10.0+cu128
CUDA available: True
CUDA device: NVIDIA GeForce RTX 4090
CUDA capability: (8, 9)
VRAM GB: 23.51


## 2. Optional package check

These are only for convenience tables/plots. The notebook does not auto-upgrade them.

In [3]:
optional_modules = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
}

missing_optional = [pip_name for mod, pip_name in optional_modules.items() if not is_installed(mod)]

if missing_optional:
    print("Missing optional packages:", missing_optional)
    print("Install manually only if needed:")
    print(sys.executable, "-m", "pip", "install", *missing_optional)
else:
    print("Optional packages available.")

try:
    import pandas as pd
except Exception:
    pd = None

try:
    import numpy as np
except Exception:
    np = None

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

Optional packages available.


## 3. Download Kaggle dataset

In [4]:
import kagglehub

dataset_cache_path = Path(kagglehub.dataset_download(KAGGLE_DATASET))
print("KaggleHub dataset path:", dataset_cache_path)

if not any(LOCAL_RAW.iterdir()):
    print("Copying dataset to:", LOCAL_RAW)
    shutil.copytree(dataset_cache_path, LOCAL_RAW, dirs_exist_ok=True)
else:
    print("Dataset already exists at:", LOCAL_RAW)

print("LOCAL_RAW:", LOCAL_RAW)

/opt/miniconda3/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.5) doesn't match a supported version!
  warnings.warn(


KaggleHub dataset path: /home/drnguyenvinh/.cache/kagglehub/datasets/nhanayai/shrimpdiseaseimagebd/versions/1
Copying dataset to: /home/drnguyenvinh/notebooks/shrimp_od_yolo_ultralytics_under12M_17models_rtx4090_v3_FIXED_OD_ONLY/kaggle_raw/shrimpdiseaseimagebd
LOCAL_RAW: /home/drnguyenvinh/notebooks/shrimp_od_yolo_ultralytics_under12M_17models_rtx4090_v3_FIXED_OD_ONLY/kaggle_raw/shrimpdiseaseimagebd


## 4. Locate annotated disease folders only

This cell intentionally does **not** search or require a Healthy folder.

In [5]:
IMG_EXTS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp",
    ".JPG", ".JPEG", ".PNG", ".BMP", ".WEBP"
}

def norm_name(s: str) -> str:
    return (
        str(s).lower()
        .replace(" ", "")
        .replace("_", "")
        .replace("-", "")
        .replace(".", "")
        .replace("(", "")
        .replace(")", "")
    )

def list_images(p: Path):
    if p is None:
        # Defensive guard: never crash on Path(None).
        return []
    p = Path(p)
    if not p.exists():
        return []
    return sorted([x for x in p.iterdir() if x.is_file() and x.suffix in IMG_EXTS])

def list_label_txts(p: Path):
    if p is None:
        return []
    p = Path(p)
    if not p.exists():
        return []
    return sorted([x for x in p.iterdir() if x.is_file() and x.suffix.lower() == ".txt"])

def find_child_dir(parent: Path, wanted_names):
    wanted = {norm_name(x) for x in wanted_names}
    for child in Path(parent).iterdir():
        if child.is_dir() and norm_name(child.name) in wanted:
            return child
    return None

def group_key_from_name(name: str):
    n = norm_name(name)

    if n in {"1bg", "bg", "blackgill", "blackgillbg"}:
        return "BG"

    if n in {"2wssv", "wssv", "whitespot", "whitespotsyndromevirus"}:
        return "WSSV"

    if n in {"4wssvbg", "wssvbg", "bgwssv", "combinationofbgandwssv", "combinationbgwssv"}:
        return "WSSV_BG"

    return None

def find_disease_group_dirs(root: Path):
    root = Path(root)
    candidates = []

    for d in root.rglob("*"):
        if not d.is_dir():
            continue

        key = group_key_from_name(d.name)
        if key is None:
            continue

        img_dir = find_child_dir(d, ["images", "Images"])
        lab_dir = find_child_dir(d, ["labels", "Labels"])

        if img_dir is None or lab_dir is None:
            continue

        n_img = len(list_images(img_dir))
        n_lab = len(list_label_txts(lab_dir))

        if n_img > 0 and n_lab > 0:
            candidates.append((key, d, img_dir, lab_dir, n_img, n_lab))

    return candidates

disease_dirs = find_disease_group_dirs(LOCAL_RAW)

print("Found annotation group dirs:")
for key, d, img, lab, n_img, n_lab in disease_dirs:
    print(f" - {key:7s} | {d} | images: {n_img} | labels: {n_lab}")

group_dirs = {}
for key, d, img, lab, n_img, n_lab in disease_dirs:
    old = group_dirs.get(key)
    if old is None or n_img > old["n_images"]:
        group_dirs[key] = {
            "root": d,
            "images": img,
            "labels": lab,
            "n_images": n_img,
            "n_labels": n_lab,
        }

required = {"BG", "WSSV", "WSSV_BG"}
missing = required - set(group_dirs)
if missing:
    raise RuntimeError(
        f"Missing annotation groups: {missing}\n"
        f"Found groups: {sorted(group_dirs.keys())}\n"
        f"LOCAL_RAW = {LOCAL_RAW}"
    )

print("\nSelected disease dirs for pure OD training:")
for k in ["BG", "WSSV", "WSSV_BG"]:
    v = group_dirs[k]
    print(f"{k:7s} => root={v['root']} | images={v['n_images']} | labels={v['n_labels']}")

print("\nHealthy handling:")
print(" - USE_HEALTHY_NEGATIVES =", USE_HEALTHY_NEGATIVES)
print(" - Healthy folder search is skipped.")
print(" - Healthy will be handled later by a classification gate/head.")

Found annotation group dirs:
 - WSSV    | /home/drnguyenvinh/notebooks/shrimp_od_yolo_ultralytics_under12M_17models_rtx4090_v3_FIXED_OD_ONLY/kaggle_raw/shrimpdiseaseimagebd/ShrimpDiseaseImageBD An Image Dataset for Computer Vision-Based Detection of Shrimp Diseases in Bangladesh/Root/Annotated Diseased Shrimp Images/Annotated Diseased Shrimp Images/2. WSSV | images: 328 | labels: 328
 - BG      | /home/drnguyenvinh/notebooks/shrimp_od_yolo_ultralytics_under12M_17models_rtx4090_v3_FIXED_OD_ONLY/kaggle_raw/shrimpdiseaseimagebd/ShrimpDiseaseImageBD An Image Dataset for Computer Vision-Based Detection of Shrimp Diseases in Bangladesh/Root/Annotated Diseased Shrimp Images/Annotated Diseased Shrimp Images/1. BG | images: 198 | labels: 198
 - WSSV_BG | /home/drnguyenvinh/notebooks/shrimp_od_yolo_ultralytics_under12M_17models_rtx4090_v3_FIXED_OD_ONLY/kaggle_raw/shrimpdiseaseimagebd/ShrimpDiseaseImageBD An Image Dataset for Computer Vision-Based Detection of Shrimp Diseases in Bangladesh/Root/A

## 5. Build pure-OD records with canonical 2-class mapping

Canonical detection classes:

```yaml
0: BG
1: WSSV
```

In [6]:
def remap_label(source_group: str, source_cls: int) -> int:
    if source_group == "BG":
        if source_cls != 0:
            raise ValueError(f"Unexpected BG source class: {source_cls}")
        return 0  # BG

    if source_group == "WSSV":
        if source_cls != 0:
            raise ValueError(f"Unexpected WSSV source class: {source_cls}")
        return 1  # WSSV

    if source_group == "WSSV_BG":
        if source_cls == 0:
            return 1  # WSSV-like region
        if source_cls == 1:
            return 0  # BG-like region

    raise ValueError(f"Unknown mapping: {source_group=} {source_cls=}")

def read_label_lines(label_path: Path, source_group: str):
    label_path = Path(label_path)
    boxes = []

    raw = label_path.read_text().strip()
    if not raw:
        return boxes

    for line_no, line in enumerate(raw.splitlines(), start=1):
        if not line.strip():
            continue

        parts = line.split()
        if len(parts) != 5:
            raise ValueError(f"Malformed label in {label_path}:{line_no}: {line}")

        source_cls = int(float(parts[0]))
        x, y, w, h = map(float, parts[1:])
        new_cls = remap_label(source_group, source_cls)

        if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
            raise ValueError(f"Invalid bbox in {label_path}:{line_no}: {line}")

        boxes.append((new_cls, x, y, w, h))

    return boxes

records = []

for group in ["BG", "WSSV", "WSSV_BG"]:
    img_dir = group_dirs[group]["images"]
    lab_dir = group_dirs[group]["labels"]

    for img_path in list_images(img_dir):
        lab_path = lab_dir / f"{img_path.stem}.txt"
        if not lab_path.exists():
            raise FileNotFoundError(f"Missing label for {img_path}")

        boxes = read_label_lines(lab_path, group)
        records.append({
            "src_image": str(img_path),
            "src_label": str(lab_path),
            "source_group": group,
            "image_level": group,
            "stem": img_path.stem,
            "n_boxes": len(boxes),
        })

# CRITICAL FIX:
# Do not iterate over HEALTHY_DIR when HEALTHY_DIR is None.
if USE_HEALTHY_NEGATIVES and HEALTHY_DIR is not None:
    for img_path in list_images(HEALTHY_DIR):
        records.append({
            "src_image": str(img_path),
            "src_label": "",
            "source_group": "Healthy",
            "image_level": "Healthy",
            "stem": img_path.stem,
            "n_boxes": 0,
        })
else:
    print("Skipping Healthy records for pure OD benchmark.")

print("Total records:", len(records))

counts = {}
box_counts = {}
for r in records:
    counts[r["image_level"]] = counts.get(r["image_level"], 0) + 1
    box_counts[r["image_level"]] = box_counts.get(r["image_level"], 0) + int(r["n_boxes"])

print("Image counts:", counts)
print("Box counts by image-level group:", box_counts)

if pd is not None:
    df_records = pd.DataFrame(records)
    display(df_records.groupby("image_level").agg(images=("src_image", "count"), boxes=("n_boxes", "sum")))
else:
    df_records = records

Skipping Healthy records for pure OD benchmark.
Total records: 746
Image counts: {'BG': 198, 'WSSV': 328, 'WSSV_BG': 220}
Box counts by image-level group: {'BG': 510, 'WSSV': 3035, 'WSSV_BG': 2024}


,images,boxes
image_level,,
BG,198,510
WSSV,328,3035
WSSV_BG,220,2024


## 6. Random stratified split by image-level group

The selected protocol is image-level random split from start to finish.

In [7]:
random.seed(SEED)

def stratified_random_split(records, train_ratio=0.8, val_ratio=0.1):
    by_group = {}
    for r in records:
        by_group.setdefault(r["image_level"], []).append(r)

    split_items = {"train": [], "val": [], "test": []}

    for group, items in by_group.items():
        items = list(items)
        random.shuffle(items)

        n = len(items)
        n_train = int(round(n * train_ratio))
        n_val = int(round(n * val_ratio))

        if n_train + n_val > n:
            n_val = max(0, n - n_train)

        split_items["train"].extend(items[:n_train])
        split_items["val"].extend(items[n_train:n_train+n_val])
        split_items["test"].extend(items[n_train+n_val:])

    for s in split_items:
        random.shuffle(split_items[s])

    return split_items

splits = stratified_random_split(records, TRAIN_RATIO, VAL_RATIO)

for split, items in splits.items():
    counts = {}
    boxes = {}
    for r in items:
        counts[r["image_level"]] = counts.get(r["image_level"], 0) + 1
        boxes[r["image_level"]] = boxes.get(r["image_level"], 0) + int(r["n_boxes"])
    print(f"\n{split}: {len(items)} images")
    print("  image counts:", counts)
    print("  box counts:", boxes)

split_meta = []
for split, items in splits.items():
    for r in items:
        rr = dict(r)
        rr["split"] = split
        split_meta.append(rr)

split_csv = TABLES_DIR / "split_metadata_disease_only.csv"
with split_csv.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["split", "src_image", "src_label", "source_group", "image_level", "stem", "n_boxes"])
    writer.writeheader()
    writer.writerows(split_meta)

print("Saved:", split_csv)


train: 596 images
  image counts: {'WSSV': 262, 'BG': 158, 'WSSV_BG': 176}
  box counts: {'WSSV': 2394, 'BG': 385, 'WSSV_BG': 1579}

val: 75 images
  image counts: {'WSSV_BG': 22, 'WSSV': 33, 'BG': 20}
  box counts: {'WSSV_BG': 220, 'WSSV': 293, 'BG': 70}

test: 75 images
  image counts: {'BG': 20, 'WSSV_BG': 22, 'WSSV': 33}
  box counts: {'BG': 55, 'WSSV_BG': 225, 'WSSV': 348}
Saved: /home/drnguyenvinh/notebooks/shrimp_od_yolo_ultralytics_under12M_17models_rtx4090_v3_FIXED_OD_ONLY/tables/split_metadata_disease_only.csv


## 7. Convert to YOLO dataset folder

In [8]:
if YOLO_DATASET_DIR.exists():
    print("Removing old converted dataset:", YOLO_DATASET_DIR)
    shutil.rmtree(YOLO_DATASET_DIR)

for split in ["train", "val", "test"]:
    (YOLO_DATASET_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (YOLO_DATASET_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

def write_converted_label(source_group: str, src_label: str, out_label_path: Path):
    out_label_path = Path(out_label_path)

    if not src_label:
        out_label_path.write_text("")
        return

    boxes = read_label_lines(Path(src_label), source_group)
    lines = [f"{cls_id} {x:.6f} {y:.6f} {w:.6f} {h:.6f}" for cls_id, x, y, w, h in boxes]
    out_label_path.write_text("\n".join(lines) + ("\n" if lines else ""))

for split, items in splits.items():
    for r in items:
        src_img = Path(r["src_image"])
        out_img = YOLO_DATASET_DIR / "images" / split / src_img.name
        out_lab = YOLO_DATASET_DIR / "labels" / split / f"{src_img.stem}.txt"

        shutil.copy2(src_img, out_img)
        write_converted_label(r["source_group"], r["src_label"], out_lab)

data_yaml = YOLO_DATASET_DIR / "data.yaml"
data_yaml.write_text(
    "path: " + str(YOLO_DATASET_DIR) + "\n"
    "train: images/train\n"
    "val: images/val\n"
    "test: images/test\n\n"
    "nc: 2\n"
    "names:\n"
    "  0: BG\n"
    "  1: WSSV\n"
)

print(data_yaml.read_text())

for split in ["train", "val", "test"]:
    imgs = list_images(YOLO_DATASET_DIR / "images" / split)
    labs = list_label_txts(YOLO_DATASET_DIR / "labels" / split)
    print(split, "images:", len(imgs), "labels:", len(labs))
    if len(imgs) != len(labs):
        raise RuntimeError(f"Image/label count mismatch in {split}: {len(imgs)} vs {len(labs)}")

    empty_labs = 0
    total_boxes = 0
    cls_counts = {0: 0, 1: 0}

    for lab in labs:
        text = lab.read_text().strip()
        if not text:
            empty_labs += 1
            continue
        for line in text.splitlines():
            parts = line.split()
            c = int(float(parts[0]))
            cls_counts[c] = cls_counts.get(c, 0) + 1
            total_boxes += 1

    print("  empty labels:", empty_labs)
    print("  total boxes:", total_boxes)
    print("  class counts:", cls_counts)

print("Converted YOLO dataset:", YOLO_DATASET_DIR)

Removing old converted dataset: /home/drnguyenvinh/notebooks/shrimp_od_yolo_ultralytics_under12M_17models_rtx4090_v3_FIXED_OD_ONLY/datasets/shrimp_od_yolo_disease_only_2cls
path: /home/drnguyenvinh/notebooks/shrimp_od_yolo_ultralytics_under12M_17models_rtx4090_v3_FIXED_OD_ONLY/datasets/shrimp_od_yolo_disease_only_2cls
train: images/train
val: images/val
test: images/test

nc: 2
names:
  0: BG
  1: WSSV

train images: 596 labels: 596
  empty labels: 0
  total boxes: 4358
  class counts: {0: 1629, 1: 2729}
val images: 75 labels: 75
  empty labels: 0
  total boxes: 583
  class counts: {0: 252, 1: 331}
test images: 75 labels: 75
  empty labels: 0
  total boxes: 628
  class counts: {0: 240, 1: 388}
Converted YOLO dataset: /home/drnguyenvinh/notebooks/shrimp_od_yolo_ultralytics_under12M_17models_rtx4090_v3_FIXED_OD_ONLY/datasets/shrimp_od_yolo_disease_only_2cls


## 8. Model list under 12M

In [9]:
MODEL_SPECS = [
    {"name": "yolov5nu", "weights": "yolov5nu.pt"},
    {"name": "yolov5su", "weights": "yolov5su.pt"},
    {"name": "yolov5n6u", "weights": "yolov5n6u.pt"},
    {"name": "yolov8n", "weights": "yolov8n.pt"},
    {"name": "yolov8s", "weights": "yolov8s.pt"},
    {"name": "yolov9t", "weights": "yolov9t.pt"},
    {"name": "yolov9s", "weights": "yolov9s.pt"},
    {"name": "yolov10n", "weights": "yolov10n.pt"},
    {"name": "yolov10s", "weights": "yolov10s.pt"},
    {"name": "yolo11n", "weights": "yolo11n.pt"},
    {"name": "yolo11s", "weights": "yolo11s.pt"},
    {"name": "yolo12n", "weights": "yolo12n.pt"},
    {"name": "yolo12s", "weights": "yolo12s.pt"},
    {"name": "yolov13n", "weights": "yolov13n.pt"},
    {"name": "yolov13s", "weights": "yolov13s.pt"},
    {"name": "yolo26n", "weights": "yolo26n.pt"},
    {"name": "yolo26s", "weights": "yolo26s.pt"},
]

for m in MODEL_SPECS:
    print(m)

{'name': 'yolov5nu', 'weights': 'yolov5nu.pt'}
{'name': 'yolov5su', 'weights': 'yolov5su.pt'}
{'name': 'yolov5n6u', 'weights': 'yolov5n6u.pt'}
{'name': 'yolov8n', 'weights': 'yolov8n.pt'}
{'name': 'yolov8s', 'weights': 'yolov8s.pt'}
{'name': 'yolov9t', 'weights': 'yolov9t.pt'}
{'name': 'yolov9s', 'weights': 'yolov9s.pt'}
{'name': 'yolov10n', 'weights': 'yolov10n.pt'}
{'name': 'yolov10s', 'weights': 'yolov10s.pt'}
{'name': 'yolo11n', 'weights': 'yolo11n.pt'}
{'name': 'yolo11s', 'weights': 'yolo11s.pt'}
{'name': 'yolo12n', 'weights': 'yolo12n.pt'}
{'name': 'yolo12s', 'weights': 'yolo12s.pt'}
{'name': 'yolov13n', 'weights': 'yolov13n.pt'}
{'name': 'yolov13s', 'weights': 'yolov13s.pt'}
{'name': 'yolo26n', 'weights': 'yolo26n.pt'}
{'name': 'yolo26s', 'weights': 'yolo26s.pt'}


## 9. Helper functions for training, resume, evaluation, and disease-level diagnosis

In [10]:
def safe_float(x, default=None):
    try:
        if x is None:
            return default
        return float(x)
    except Exception:
        return default

def model_file_size_mb(path):
    path = Path(path)
    if not path.exists():
        return None
    return path.stat().st_size / (1024**2)

def count_model_params(model_obj):
    try:
        return sum(p.numel() for p in model_obj.model.parameters()) / 1e6
    except Exception:
        return None

def get_run_name(model_name):
    return f"{model_name}_shrimpOD_2cls_img{IMG_SIZE}_ep{EPOCHS}_seed{SEED}_diseaseOnly"

def get_run_dir(model_name):
    return RUNS_DIR / get_run_name(model_name)

def get_best_last_paths(model_name):
    run_dir = get_run_dir(model_name)
    weights_dir = run_dir / "weights"
    return weights_dir / "best.pt", weights_dir / "last.pt"

def write_error(model_name, stage, err):
    ERROR_DIR.mkdir(parents=True, exist_ok=True)
    out = ERROR_DIR / f"{model_name}_{stage}_error.txt"
    out.write_text(traceback.format_exc())
    print(f"[ERROR] {model_name} {stage}: {err}")
    print("Saved traceback:", out)

def extract_box_metrics(results):
    out = {}
    box = getattr(results, "box", None)
    if box is not None:
        out["precision_mp"] = safe_float(getattr(box, "mp", None))
        out["recall_mr"] = safe_float(getattr(box, "mr", None))
        out["map50"] = safe_float(getattr(box, "map50", None))
        out["map50_95"] = safe_float(getattr(box, "map", None))

        maps = getattr(box, "maps", None)
        if maps is not None:
            try:
                maps_list = list(maps)
                out["ap_BG"] = safe_float(maps_list[0]) if len(maps_list) > 0 else None
                out["ap_WSSV"] = safe_float(maps_list[1]) if len(maps_list) > 1 else None
            except Exception:
                out["ap_BG"] = None
                out["ap_WSSV"] = None

    speed = getattr(results, "speed", None)
    if isinstance(speed, dict):
        out["speed_preprocess_ms"] = safe_float(speed.get("preprocess"))
        out["speed_inference_ms"] = safe_float(speed.get("inference"))
        out["speed_postprocess_ms"] = safe_float(speed.get("postprocess"))
        total = 0.0
        ok = False
        for k in ["preprocess", "inference", "postprocess"]:
            v = speed.get(k)
            if v is not None:
                total += float(v)
                ok = True
        out["speed_total_ms"] = total if ok else None
        out["fps_total"] = (1000.0 / total) if ok and total > 0 else None

    return out

def labels_from_yolo_txt(label_path):
    labels = set()
    text = Path(label_path).read_text().strip()
    if not text:
        return labels
    for line in text.splitlines():
        parts = line.split()
        if not parts:
            continue
        labels.add(int(float(parts[0])))
    return labels

def image_level_from_label_set(label_set):
    if 0 in label_set and 1 in label_set:
        return "WSSV_BG"
    if 0 in label_set:
        return "BG"
    if 1 in label_set:
        return "WSSV"
    return "NO_DET"

def macro_f1_manual(y_true, y_pred, labels):
    f1s = []
    per_label = {}
    for lab in labels:
        tp = sum(1 for t, p in zip(y_true, y_pred) if t == lab and p == lab)
        fp = sum(1 for t, p in zip(y_true, y_pred) if t != lab and p == lab)
        fn = sum(1 for t, p in zip(y_true, y_pred) if t == lab and p != lab)
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) else 0.0
        f1s.append(f1)
        per_label[lab] = {"precision": prec, "recall": rec, "f1": f1, "tp": tp, "fp": fp, "fn": fn}
    return (sum(f1s) / len(f1s) if f1s else 0.0), per_label

def accuracy_manual(y_true, y_pred):
    return sum(1 for t, p in zip(y_true, y_pred) if t == p) / len(y_true) if y_true else 0.0

def kappa_manual(y_true, y_pred, labels):
    n = len(y_true)
    if n == 0:
        return None
    po = accuracy_manual(y_true, y_pred)
    pe = 0.0
    for lab in labels:
        p_true = sum(1 for t in y_true if t == lab) / n
        p_pred = sum(1 for p in y_pred if p == lab) / n
        pe += p_true * p_pred
    if abs(1 - pe) < 1e-12:
        return None
    return (po - pe) / (1 - pe)

def evaluate_image_level_from_predictions(model, model_name, conf=0.25, iou=0.7):
    test_img_dir = YOLO_DATASET_DIR / "images" / "test"
    test_lab_dir = YOLO_DATASET_DIR / "labels" / "test"
    test_images = list_images(test_img_dir)

    y_true, y_pred = [], []
    pred_rows = []

    if not test_images:
        return {}

    results_iter = model.predict(
        source=[str(x) for x in test_images],
        imgsz=IMG_SIZE,
        conf=conf,
        iou=iou,
        max_det=300,
        device=DEVICE,
        verbose=False,
        stream=True,
    )

    for img_path, res in zip(test_images, results_iter):
        true_set = labels_from_yolo_txt(test_lab_dir / f"{img_path.stem}.txt")
        true_level = image_level_from_label_set(true_set)

        pred_set = set()
        try:
            if res.boxes is not None and len(res.boxes) > 0:
                cls_vals = res.boxes.cls.detach().cpu().numpy().tolist()
                pred_set = {int(c) for c in cls_vals}
        except Exception:
            pred_set = set()

        pred_level = image_level_from_label_set(pred_set)

        y_true.append(true_level)
        y_pred.append(pred_level)

        pred_rows.append({
            "image": str(img_path),
            "true": true_level,
            "pred": pred_level,
            "pred_classes": ",".join(map(str, sorted(pred_set))) if pred_set else "",
        })

    labels_core = ["BG", "WSSV", "WSSV_BG"]
    labels_all = ["BG", "WSSV", "WSSV_BG", "NO_DET"]

    macro_f1, per_label = macro_f1_manual(y_true, y_pred, labels_core)
    acc = accuracy_manual(y_true, y_pred)
    kappa = kappa_manual(y_true, y_pred, labels_all)

    n = len(y_true)
    no_det = sum(1 for p in y_pred if p == "NO_DET")
    disease_miss_rate = no_det / n if n else None

    out_pred_csv = TABLES_DIR / f"{model_name}_test_image_level_predictions.csv"
    with out_pred_csv.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["image", "true", "pred", "pred_classes"])
        writer.writeheader()
        writer.writerows(pred_rows)

    return {
        "diagnosis_accuracy_disease_only": acc,
        "diagnosis_macro_f1_disease_only": macro_f1,
        "diagnosis_kappa_disease_only": kappa,
        "disease_miss_rate_no_det": disease_miss_rate,
        "recall_BG_diag": per_label.get("BG", {}).get("recall"),
        "recall_WSSV_diag": per_label.get("WSSV", {}).get("recall"),
        "recall_WSSV_BG_diag": per_label.get("WSSV_BG", {}).get("recall"),
        "image_level_pred_csv": str(out_pred_csv),
    }

## 10. Train and evaluate all models

Resume policy:
1. If `weights/best.pt` exists: skip training and evaluate.
2. Else if `weights/last.pt` exists: resume from `last.pt`.
3. Else start training from pretrained model name.
4. Any model failure is logged and the loop continues.

In [ ]:
summary_rows = []
live_csv = TABLES_DIR / "model_comparison_live.csv"

def save_summary(rows, path):
    if not rows:
        return
    fieldnames = sorted({k for r in rows for k in r.keys()})
    with Path(path).open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

for spec in MODEL_SPECS:
    model_name = spec["name"]
    weights = spec["weights"]
    run_name = get_run_name(model_name)
    best_pt, last_pt = get_best_last_paths(model_name)

    print("\n" + "=" * 100)
    print("MODEL:", model_name, "| weights:", weights)
    print("RUN:", run_name)
    print("best:", best_pt)
    print("last:", last_pt)

    row = {
        "model": model_name,
        "weights": weights,
        "run_name": run_name,
        "status": "unknown",
        "epochs": EPOCHS,
        "imgsz": IMG_SIZE,
        "dataset": str(YOLO_DATASET_DIR),
        "timestamp": datetime.now().isoformat(timespec="seconds"),
    }

    try:
        if best_pt.exists():
            print("Found best.pt; skipping training.")
            row["train_action"] = "skip_existing_best"
        elif last_pt.exists():
            print("Found last.pt; resuming training.")
            model = YOLO(str(last_pt))
            model.train(resume=True)
            row["train_action"] = "resume_last"
        else:
            print("Starting new training.")
            model = YOLO(weights)
            model.train(
                data=str(data_yaml),
                epochs=EPOCHS,
                imgsz=IMG_SIZE,
                batch=-1,
                device=DEVICE,
                workers=WORKERS,
                seed=SEED,
                deterministic=True,
                pretrained=True,
                optimizer="auto",
                cos_lr=True,
                close_mosaic=10,
                mosaic=1.0,
                mixup=0.0,
                copy_paste=0.0,
                degrees=5.0,
                translate=0.05,
                scale=0.30,
                shear=0.0,
                perspective=0.0,
                fliplr=0.5,
                flipud=0.0,
                hsv_h=0.015,
                hsv_s=0.5,
                hsv_v=0.3,
                amp=True,
                cache=False,
                val=True,
                plots=True,
                project=str(RUNS_DIR),
                name=run_name,
                exist_ok=True,
            )
            row["train_action"] = "train_new"

    except Exception as e:
        row["status"] = "train_failed"
        row["error"] = str(e)
        write_error(model_name, "train", e)
        summary_rows.append(row)
        save_summary(summary_rows, live_csv)
        continue

    best_pt, last_pt = get_best_last_paths(model_name)
    if best_pt.exists():
        eval_weights = best_pt
    elif last_pt.exists():
        print("best.pt not found; using last.pt for evaluation.")
        eval_weights = last_pt
    else:
        row["status"] = "missing_weights_after_train"
        summary_rows.append(row)
        save_summary(summary_rows, live_csv)
        continue

    row["eval_weights"] = str(eval_weights)
    row["model_size_mb"] = model_file_size_mb(eval_weights)

    try:
        eval_model = YOLO(str(eval_weights))
        row["params_m"] = count_model_params(eval_model)

        val_results = eval_model.val(
            data=str(data_yaml),
            split="test",
            imgsz=IMG_SIZE,
            batch=16,
            device=DEVICE,
            workers=WORKERS,
            plots=True,
            save_json=False,
            project=str(EVAL_DIR),
            name=f"{run_name}_test",
            exist_ok=True,
        )

        row.update(extract_box_metrics(val_results))
        row.update(evaluate_image_level_from_predictions(eval_model, model_name, conf=0.25, iou=0.7))

        map_ = row.get("map50_95") or 0.0
        f1_ = row.get("diagnosis_macro_f1_disease_only") or 0.0
        fps_ = row.get("fps_total") or 0.0
        size_ = row.get("model_size_mb") or 999.0
        miss_ = row.get("disease_miss_rate_no_det") or 0.0

        row["mobile_edge_score"] = (
            0.40 * map_
            + 0.30 * f1_
            + 0.20 * min(fps_ / 200.0, 1.0)
            + 0.10 * max(0.0, 1.0 - size_ / 50.0)
            - 0.20 * miss_
        )

        row["status"] = "ok"

    except Exception as e:
        row["status"] = "eval_failed"
        row["error"] = str(e)
        write_error(model_name, "eval", e)

    summary_rows.append(row)
    save_summary(summary_rows, live_csv)
    print("Saved live summary:", live_csv)

print("\nFinished loop.")
print("Live CSV:", live_csv)


MODEL: yolov5nu | weights: yolov5nu.pt
RUN: yolov5nu_shrimpOD_2cls_img1024_ep30_seed42_diseaseOnly
best: /home/drnguyenvinh/notebooks/shrimp_od_yolo_ultralytics_under12M_17models_rtx4090_v3_FIXED_OD_ONLY/runs_train_30ep/yolov5nu_shrimpOD_2cls_img1024_ep30_seed42_diseaseOnly/weights/best.pt
last: /home/drnguyenvinh/notebooks/shrimp_od_yolo_ultralytics_under12M_17models_rtx4090_v3_FIXED_OD_ONLY/runs_train_30ep/yolov5nu_shrimpOD_2cls_img1024_ep30_seed42_diseaseOnly/weights/last.pt
Starting new training.
Ultralytics 8.4.67 🚀 Python-3.13.2 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24072MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/home/drnguyenvinh/notebooks/shrimp_od_yolo_ultralytics_under12M_17models_rtx4090_v3_FIXED_OD_ONLY/da

## 11. Final table and ranking

In [ ]:
final_csv = TABLES_DIR / "final_model_comparison.csv"
final_md = TABLES_DIR / "final_model_comparison.md"

if pd is not None and live_csv.exists():
    df = pd.read_csv(live_csv)

    if "mobile_edge_score" in df.columns:
        df = df.sort_values(["status", "mobile_edge_score"], ascending=[True, False])
    elif "map50_95" in df.columns:
        df = df.sort_values(["status", "map50_95"], ascending=[True, False])

    df.to_csv(final_csv, index=False)
    try:
        final_md.write_text(df.to_markdown(index=False))
    except Exception:
        final_md.write_text(df.to_string(index=False))

    display_cols = [
        "model", "status", "map50", "map50_95", "precision_mp", "recall_mr",
        "ap_BG", "ap_WSSV",
        "diagnosis_macro_f1_disease_only", "diagnosis_accuracy_disease_only",
        "disease_miss_rate_no_det",
        "recall_BG_diag", "recall_WSSV_diag", "recall_WSSV_BG_diag",
        "speed_total_ms", "fps_total", "model_size_mb", "params_m", "mobile_edge_score",
        "train_action"
    ]
    display_cols = [c for c in display_cols if c in df.columns]

    print("Final CSV:", final_csv)
    print("Final MD:", final_md)
    display(df[display_cols])
else:
    print("Pandas unavailable or live CSV missing.")
    print("Live CSV:", live_csv)

## 12. Optional plots

In [ ]:
if pd is not None and plt is not None and final_csv.exists():
    df = pd.read_csv(final_csv)
    ok = df[df["status"] == "ok"].copy() if "status" in df.columns else df.copy()

    if len(ok) > 0:
        if "fps_total" in ok.columns and "map50_95" in ok.columns:
            plt.figure(figsize=(8, 5))
            plt.scatter(ok["fps_total"], ok["map50_95"])
            for _, r in ok.iterrows():
                plt.annotate(str(r["model"]), (r["fps_total"], r["map50_95"]), fontsize=8)
            plt.xlabel("FPS total")
            plt.ylabel("mAP50-95")
            plt.title("mAP50-95 vs FPS")
            plt.grid(True, alpha=0.3)
            out = TABLES_DIR / "plot_map50_95_vs_fps.png"
            plt.savefig(out, dpi=200, bbox_inches="tight")
            plt.show()
            print("Saved:", out)

        if "model_size_mb" in ok.columns and "diagnosis_macro_f1_disease_only" in ok.columns:
            plt.figure(figsize=(8, 5))
            plt.scatter(ok["model_size_mb"], ok["diagnosis_macro_f1_disease_only"])
            for _, r in ok.iterrows():
                plt.annotate(str(r["model"]), (r["model_size_mb"], r["diagnosis_macro_f1_disease_only"]), fontsize=8)
            plt.xlabel("Model size MB")
            plt.ylabel("Disease-only diagnosis Macro-F1")
            plt.title("Diagnosis Macro-F1 vs Model Size")
            plt.grid(True, alpha=0.3)
            out = TABLES_DIR / "plot_diag_f1_vs_size.png"
            plt.savefig(out, dpi=200, bbox_inches="tight")
            plt.show()
            print("Saved:", out)
    else:
        print("No successful model rows to plot.")
else:
    print("Skipping plots because pandas/matplotlib/final CSV is unavailable.")

## 13. Next phase: Healthy classification gate

This notebook intentionally does not use Healthy in OD.

Next notebook should train a lightweight classifier:

- Healthy
- BG
- WSSV
- WSSV_BG

Then build:

```text
Input image
  ↓
Healthy classifier gate
  ├── Healthy high confidence -> output Healthy, skip OD
  └── Diseased/uncertain -> run best OD model
```